### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.
We will be provided a company name and their primary website.

In [ ]:
# type: ignore
import os
import requests 
import json
import ollama
from typing import List
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display,clear_output

In [ ]:
# constants
MODEL = "llama3.2"

In [ ]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [14]:
site = Website("https://www.riotgames.com/en")
site.links

['#content',
 '#riotbar-bar',
 '/en/news',
 '/en/news/the-tech-behind-swarm',
 '/en/news/know-before-you-go-2xko-at-evo-2025',
 '/en/news/2xko-closed-beta-announcement',
 '/en/news/map-design-in-valorant-super-art-power-hour-ep-9',
 '/en/news/riotling-day-2025',
 '/en/news',
 'https://www.leagueoflegends.com/',
 'https://playvalorant.com/',
 'https://teamfighttactics.leagueoflegends.com/',
 'https://wildrift.leagueoflegends.com/',
 'https://playruneterra.com/',
 'https://2xko.riotgames.com/',
 'https://riftbound.leagueoflegends.com/',
 'https://lolesports.com/',
 'https://valorantesports.com/',
 'https://www.arcane.com/',
 'https://www.youtube.com/c/riotgamesmusic',
 '/en/work-with-us',
 '/en/work-with-us/offices',
 'https://www.riotgames.com/en/work-with-us',
 '/',
 '/who-we-are',
 '/work-with-us',
 '/news',
 '/en/press',
 '/en/security',
 '/en/legal',
 '/en/leadership',
 '/en/candidate-privacy',
 '/en/terms-of-service',
 '/en/privacy-notice',
 'https://support.riotgames.com/hc/en-us'

In [15]:
link_system_prompt = """You are an expert web content analyst. Your task is to intelligently filter a list of URLs from a company's website to find the pages that contain the most valuable information for creating a company brochure. The brochure will be for **prospective clients, potential investors, and potential recruits**.

You will be given a base URL and a list of links found on that page. Some links may be relative paths.

**Your instructions are:**

1.  **Analyze the list of links** and select ONLY the ones most likely to contain substantive information about the company's mission, products, services, history, culture, or career opportunities.
2.  **Prioritize links** with keywords like: `about`, `company`, `who we are`, `mission`, `careers`, `jobs`, `investors`, `news`, `press`, `products`, `solutions`.
3.  **Explicitly EXCLUDE** links related to:
    * Social media (e.g., twitter.com, facebook.com, linkedin.com)
    * User login or sign-up pages
    * Legal documents (privacy policy, terms of service)
    * Customer support or contact forms (unless it's a general "Contact Us" page)
    * Links to external, unrelated websites.
4.  **Construct Full URLs:** For any relative links (e.g., `/about-us`), you MUST construct the full, absolute URL by prepending the provided base URL. For example, if the base URL is `https://www.example.com` and the link is `/about`, the full URL is `https://www.example.com/about`.
5.  **Respond ONLY with a single JSON object.** Do not add any introductory text or explanations. The JSON object must have a single key, `"links"`, which contains a list of objects. Each object in the list must have two keys:
    * `"type"`: A short, descriptive, lowercase label for the page (e.g., "about page", "careers page").
    * `"url"`: The full, absolute URL.

**Example of the required JSON output format:**

```json
{
  "links": [
    {
      "type": "about page",
      "url": "[https://www.example.com/company/about-us](https://www.example.com/company/about-us)"
    },
    {
      "type": "careers page",
      "url": "[https://www.example.com/careers](https://www.example.com/careers)"
    },
    {
      "type": "investor relations",
      "url": "[https://investors.example.com/](https://investors.example.com/)"
    }
  ]
}
```"""

In [16]:
print(link_system_prompt)

You are an expert web content analyst. Your task is to intelligently filter a list of URLs from a company's website to find the pages that contain the most valuable information for creating a company brochure. The brochure will be for **prospective clients, potential investors, and potential recruits**.

You will be given a base URL and a list of links found on that page. Some links may be relative paths.

**Your instructions are:**

1.  **Analyze the list of links** and select ONLY the ones most likely to contain substantive information about the company's mission, products, services, history, culture, or career opportunities.
2.  **Prioritize links** with keywords like: `about`, `company`, `who we are`, `mission`, `careers`, `jobs`, `investors`, `news`, `press`, `products`, `solutions`.
3.  **Explicitly EXCLUDE** links related to:
    * Social media (e.g., twitter.com, facebook.com, linkedin.com)
    * User login or sign-up pages
    * Legal documents (privacy policy, terms of servic

In [17]:
from urllib.parse import urlparse

def get_links_user_prompt(website):
    # Extract the base URL to help the model construct full URLs from relative paths.
    parsed_url = urlparse(website.url)
    base_url = f"{parsed_url.scheme}://{parsed_url.netloc}"

    user_prompt = f"The following list of links was found on the website {website.url}. The base URL for constructing full links is {base_url}.\n"
    user_prompt += "Please identify the links most relevant for a company brochure (e.g., about, careers, investors, products) and respond with the full URLs in the required JSON format.\n"
    user_prompt += "Links:\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [18]:
print(get_links_user_prompt(site))

The following list of links was found on the website https://www.riotgames.com/en. The base URL for constructing full links is https://www.riotgames.com.
Please identify the links most relevant for a company brochure (e.g., about, careers, investors, products) and respond with the full URLs in the required JSON format.
Links:
#content
#riotbar-bar
/en/news
/en/news/the-tech-behind-swarm
/en/news/know-before-you-go-2xko-at-evo-2025
/en/news/2xko-closed-beta-announcement
/en/news/map-design-in-valorant-super-art-power-hour-ep-9
/en/news/riotling-day-2025
/en/news
https://www.leagueoflegends.com/
https://playvalorant.com/
https://teamfighttactics.leagueoflegends.com/
https://wildrift.leagueoflegends.com/
https://playruneterra.com/
https://2xko.riotgames.com/
https://riftbound.leagueoflegends.com/
https://lolesports.com/
https://valorantesports.com/
https://www.arcane.com/
https://www.youtube.com/c/riotgamesmusic
/en/work-with-us
/en/work-with-us/offices
https://www.riotgames.com/en/work-w

In [19]:
def get_links(url):
    website = Website(url)
    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
        ], format = "json"  #Define format as json!
    )
    result = response['message']['content']

    return json.loads(result)


In [20]:
# I'm using HuggingFace..

huggingface = Website("https://huggingface.co")
huggingface.links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 'inference/get-started',
 '/spaces',
 '/models',
 '/Qwen/Qwen3-Coder-480B-A35B-Instruct',
 '/zai-org/GLM-4.5',
 '/bosonai/higgs-audio-v2-generation-3B-base',
 '/tencent/HunyuanWorld-1',
 '/Qwen/Qwen3-235B-A22B-Instruct-2507',
 '/models',
 '/spaces/enzostvs/deepsite',
 '/spaces/smola/higgs_audio_v2',
 '/spaces/zumjoy/Multi-Style_Video-to-Anime_Generator',
 '/spaces/Qwen/Qwen3-Coder-WebDev',
 '/spaces/black-forest-labs/FLUX.1-Kontext-Dev',
 '/spaces',
 '/datasets/fka/awesome-chatgpt-prompts',
 '/datasets/interstellarninja/hermes_reasoning_tool_use',
 '/datasets/NousResearch/Hermes-3-Dataset',
 '/datasets/MegaScience/MegaScience',
 '/datasets/microsoft/rStar-Coder',
 '/datasets',
 '/join',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/allenai',
 '/facebook',
 '/a

### Making the brochure

In [21]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [22]:
print(get_all_details("https://huggingface.co"))

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/docs'}, {'type': 'careers page', 'url': 'https://huggingface.co/careers'}, {'type': 'investor relations', 'url': 'https://endpoints.huggingface.co'}, {'type': 'products page', 'url': 'https://huggingface.co/models'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}]}
Landing page:
Webpage Title:
Hugging Face – The AI community building the future.
Webpage Contents:
Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
NEW
Get started with Inference in seconds 🚀
Reachy Mini: The Open Robot for AI Builders
Welcome Cohere on the Hub 🔥
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 1M+ models
Trending on
this week
Models
Qwen/Qwen3-Coder-480B-A35B-Instruct
Updated
6 days ago
•
14.4k
•
871
zai-org/GLM-4.5
Updated
1 day ago
•
919
•
547
bosonai/higgs-a